<img src="https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/IMPERIAL_logo_RGB_Blue_2024.svg" alt="Imperial Logo" width="500"/><br /><br />

Manipulating and Visualising Data — Solutions
==============
### SURG70098 - Surgical Data Science and AI
### Stuart Bowyer

*Worked solutions to the **» Predict** and **» Your Turn** tasks in **02_Manipulating_and_Visualising_Data.ipynb**.* These are deliberately in a **separate file** so the answers are not sitting next to the problems — try each task first. Solutions are written as **explanations**, not just answer keys.

## Functions

### Predict: arguments the other way round
**A wrong number, with no error.** `calculate_bmi(90, 1.80)` runs with `height_m = 90` and `weight_kg = 1.8`, giving `1.8 / 8100 = 0.00022`. Python matches positional arguments by **order**, not by name, and a number is a number whichever parameter it lands in. A silent wrong answer is the worst outcome, which is why keyword arguments, `calculate_bmi(weight_kg=90, height_m=1.8)`, are worth the typing when the order is not obvious.

In [ ]:
def calculate_bmi(height_m, weight_kg):
    return weight_kg / height_m ** 2

print(calculate_bmi(90, 1.80))                        # 0.000222...: nonsense, no error
print(calculate_bmi(weight_kg=90, height_m=1.80))     # 27.78

### Predict: `print(bmi)` after the call
**`NameError: name 'bmi' is not defined`.** `bmi` is a **local** variable: it is created when the function runs and discarded when it returns. Only the returned value gets out, and only if the caller keeps it (`result = ...`). The first `print(result)` works for exactly that reason.

In [ ]:
def calculate_bmi(height_m, weight_kg):
    bmi = weight_kg / height_m ** 2
    return bmi

result = calculate_bmi(1.80, 90)
print(result)
# print(bmi) would raise NameError

### Your Turn: mean arterial pressure
**Approach:** two parameters, the formula in the body, `return` the result. The parameter names are yours to choose; `systolic` and `diastolic` make the call readable.

In [ ]:
def calculate_map(systolic, diastolic):
    """Mean arterial pressure (mmHg) from systolic and diastolic pressure (mmHg)."""
    map_mmhg = diastolic + (systolic - diastolic) / 3
    return map_mmhg

print(calculate_map(120, 80))   # 93.33...

### Your Turn: BMI category (independent)
**Approach:** an `if` / `elif` / `else` chain checked from the lowest threshold up, each branch returning a string. Because each `return` ends the function, the chain is read top to bottom and the first true condition wins, so the upper bounds are implied. The result of one function becomes the argument of the next.

**The stretch:** type hints go on the parameter and the return (`bmi: float` and `-> str`); the docstring is the first line of the body. For the input checks, plausible ranges catch both mistakes at once: a height outside 0.5 to 2.5 m is either centimetres or a swapped argument, and a weight outside 2 to 400 kg likewise. The check goes **first**, before the calculation. Within what we know so far, the function can print what is wrong and return `None`; do not silently convert 180 to 1.80, because guessing hides the error. (The idiomatic Python is `raise ValueError(...)`, which stops the program with your message; you will meet `raise` later.)

In [ ]:
def calculate_bmi(height_m: float, weight_kg: float) -> float:
    """Calculate BMI from height in metres and weight in kilograms."""
    if not 0.5 <= height_m <= 2.5:
        print('Height', height_m, 'is not plausible in metres')
        return None
    if not 2 <= weight_kg <= 400:
        print('Weight', weight_kg, 'is not plausible in kilograms')
        return None
    return weight_kg / height_m ** 2

def bmi_category(bmi: float) -> str:
    """Return the NHS BMI category for a BMI value."""
    if bmi < 18.5:
        return 'underweight'
    elif bmi < 25:
        return 'healthy'
    elif bmi < 30:
        return 'overweight'
    else:
        return 'obese'

bmi = calculate_bmi(1.80, 90)
print(round(bmi, 1), bmi_category(bmi))

print(calculate_bmi(180, 90))      # centimetres by mistake
print(calculate_bmi(90, 1.80))     # arguments the wrong way round

### Predict: four imports, four calls
* **A works:** `import math` gives the name `math`, and `math.sqrt(16)` is `4.0`.
* **B errors:** `NameError: name 'sqrt' is not defined`. Importing the module does not spill its contents into your namespace; you reach them through `math.`.
* **C works:** `from statistics import mean` gives you the bare name `mean`.
* **D errors:** `NameError: name 'statistics' is not defined`. The alias replaces the name: only `stats` exists.

The rule: the usable name is exactly what is written after `import` or after `as`. This is why code copied from a tutorial fails when the tutorial wrote `pandas.read_csv` and you imported `pandas as pd`, or the reverse.

## CRP results for one patient, with lists
**Approach:** one loop over the indices, since the two lists line up by position. For the count, add one each time the patient matches, with the count started at 0 before the loop. For the mean, also keep a running total: take the text before the unit (`replace` or a slice), cast to `float`, add it. Return total over count.

**The trap:** forgetting to cast, which makes `+` join text instead of adding numbers, or dividing by `len(crp)` rather than that patient's count. Twenty values were typed into the cell; nobody types 200, let alone 200,000, which is one reason for pandas. The other is that calling this for every patient runs the loop four times over, where pandas does it in one `groupby`.

In [ ]:
patients = ['Yusuf', 'Amara', 'Noah', 'Priya', 'Noah', 'Noah', 'Amara', 'Priya', 'Noah', 'Noah',
            'Yusuf', 'Noah', 'Priya', 'Amara', 'Amara', 'Priya', 'Yusuf', 'Yusuf', 'Yusuf', 'Yusuf']
crp = ['0.1912 mg/L', '0.0697 mg/L', '6.063 mg/L', '8.7888 mg/L', '6.1414 mg/L', '5.8497 mg/L', '2.4352 mg/L',
       '2.4059 mg/L', '7.8755 mg/L', '9.4961 mg/L', '0.1357 mg/L', '4.4206 mg/L', '10.7717 mg/L', '0.0268 mg/L',
       '0.0281 mg/L', '4.0544 mg/L', '0.0382 mg/L', '0.0034 mg/L', '0.6772 mg/L', '1.7951 mg/L']

def count_results(crp, patients, name):
    count = 0
    for index in range(len(crp)):
        if patients[index] == name:
            count = count + 1
    return count

print(count_results(crp, patients, 'Noah'))   # 6

def average_crp(crp, patients, name):
    total = 0
    count = 0
    for index in range(len(crp)):
        if patients[index] == name:
            value = float(crp[index].replace(' mg/L', ''))
            total = total + value
            count = count + 1
    return total / count

print(average_crp(crp, patients, 'Noah'))     # 6.64

for name in ['Amara', 'Noah', 'Priya', 'Yusuf']:
    print(name, round(average_crp(crp, patients, name), 2))   # Noah is highest

## Tables in pandas

### Every height in centimetres
**Approach:** `df['height_m'] * 100`. It gives **five numbers**: `175.0, 180.0, 162.0, 168.0, 171.0`, each height multiplied, as a Series. A Python list repeats under `*`; a pandas Series does the arithmetic **elementwise**, one result per row, with no loop. Everything else in pandas rests on this: a comparison on a column gives one `True`/`False` per row, which is exactly what a filter needs.

## Independent practice

### Predict: selecting rows and values
1. `df['age']` is the whole column: `67, 45, 82, 59, 73`.
2. `df.loc[2]` is the row labelled 2: patient `1003`, aged 82, 58 kg, 1.62 m, Medical, admitted 2026-01-06, plus the `bmi` of 22.1 and `overweight` of `False` added two slides earlier.
3. `df.loc[4, 'ward']` is one value, by row label and column name: `'Medical'`.
4. `df.loc[1:3]` is **three** rows, labelled 1, 2 and 3.

**The trap:** a `.loc` slice **includes its end label**, unlike the list and string slices of last session, where `[1:3]` gave two items. Labels are names, not positions, and pandas takes "from 1 to 3" literally. (`.iloc[1:3]` behaves like a list slice and gives two.)

### Predict: `df[df['ward'] == 'Surgical']`
**Three rows**: patients 1001, 1002 and 1004. The comparison produces `True, True, False, True, False`, and the brackets keep the `True` rows. Row labels are kept as they were, so the result is labelled 0, 1 and 3.

### Predict: a filter that matches nothing
**An empty DataFrame**, printed as the column names with no rows. No error. The boolean Series `df['age'] > 100` is all `False`, and selecting with all-`False` keeps nothing. This is why `print(df.shape)` after a filter is a habit worth having: zero rows, or a suspiciously small number, is the only sign of a wrong condition.

### Your Turn: elevated CRP
**Approach:** `head()` to look; the condition `df_crp['crp_value'] > 5.0` as the filter; `.shape` for the count. The `.str.removesuffix(...).astype(float)` line is the whole of the next block in miniature.

In [ ]:
import pandas as pd
DATA = "https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/data/"

df_crp = pd.read_csv(DATA + "crp_results.csv")
print(df_crp.head())

df_crp['crp_value'] = df_crp['crp'].str.removesuffix(' mg/L').astype(float)

elevated = df_crp[df_crp['crp_value'] > 5.0]

print(elevated.shape)    # (44, 3): 44 elevated results
elevated

### Your Turn: load `str_data.csv`
**Approach:** `pd.read_csv`, then `.shape`, `.dtypes`, `.head()`. Only `patient_id` is a number. `hr`, `bp` and `conscious_level` contain numbers but are text (`str` or `object`), because of the units and labels, and `microbio_culture` is text proper. Cleaning them is the next section.

In [ ]:
import pandas as pd
DATA = "https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/data/"

df = pd.read_csv(DATA + "str_data.csv")

print(df.shape)     # (2000, 5)
print(df.dtypes)
df.head()

### Predict: `.mean()` on `hr_clean`
**A `TypeError`.** Removing the unit leaves text that *looks* like a number, and there is no mean of text. The exact message depends on the pandas version, but it will mention strings or converting to numeric. The fix is `.astype(float)`.

### Your Turn: systolic, diastolic and pulse pressure
**Approach:** with the unit gone, `split('/')` gives `['96', '72']` for every row; `.str[0]` and `.str[1]` take the two parts; `.astype(int)` makes them numbers; then subtract the two columns. If you cast *before* removing the unit, the diastolic cast fails on `'72 mmHg'`.

In [ ]:
import pandas as pd
DATA = "https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/data/"

df = pd.read_csv(DATA + "str_data.csv")

bp = df['bp'].str.removesuffix(' mmHg')

df['systolic'] = bp.str.split('/').str[0].astype(int)
df['diastolic'] = bp.str.split('/').str[1].astype(int)
df['pulse_pressure'] = df['systolic'] - df['diastolic']
df.head()

### Your Turn: mean, median and standard deviation (independent)
**Approach:** the three methods on the column, `median()` being the new one. Pulse pressure is close to symmetric here, so the median is near the mean; a big gap between them is the first sign of a skewed distribution or outliers.

In [ ]:
import pandas as pd
DATA = "https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/data/"

df = pd.read_csv(DATA + "str_data.csv")

bp = df['bp'].str.removesuffix(' mmHg')
df['pulse_pressure'] = bp.str.split('/').str[0].astype(int) - bp.str.split('/').str[1].astype(int)

print('mean:  ', round(df['pulse_pressure'].mean(), 2))
print('median:', df['pulse_pressure'].median())
print('st dev:', round(df['pulse_pressure'].std(), 2))

### Your Turn: elevated results per patient
**Approach:** the comparison makes a boolean column; group by `patients`; `sum()` counts the `True` values. Priya 17, Noah 24, Oliver 3, and zero for Amara and Yusuf, who appear with 0 because every patient forms a group.

In [ ]:
import pandas as pd
DATA = "https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/data/"

df_crp = pd.read_csv(DATA + "crp_results.csv")
df_crp['crp_value'] = df_crp['crp'].str.removesuffix(' mg/L').astype(float)

df_crp['elevated'] = df_crp['crp_value'] > 5.0
df_crp.groupby('patients')['elevated'].sum()

## Combining tables

### Predict: concatenating tables with different columns
**A table, no error.** Columns are matched by name: the result has all seven column names, and every cell that an input did not supply is `NaN`. The two demographics rows have `NaN` for `recovery_days`; the two recovery rows have `NaN` for everything but `id` and `recovery_days`. Only the `id` column is shared, and even that is not the same patient. A concat of the wrong tables looks like success until you look at the result.

### Predict: merging with a 10-row sample
**10 rows.** The default merge is `inner`: a row survives only if its key is in **both** tables, and only ten `id`s are. The other 990 patients are dropped without a message, which is the single most common way to lose data in a pandas pipeline.

## Plotting

### Your Turn: two patients' CRP
**Approach:** `label` names each series for the legend; the second `plot` takes `hours, crp_b`; `xlabel` takes a string. `plt.legend()` draws the box using the labels you gave. Patient A's CRP peaks at a day and settles, the expected course; Patient B's is still rising at 72 hours, which is the pattern that suggests a complication such as a leak.

In [ ]:
import matplotlib.pyplot as plt

hours = [0, 6, 12, 24, 48, 72]
crp_a = [5, 40, 95, 150, 120, 60]
crp_b = [8, 60, 140, 210, 250, 230]

plt.plot(hours, crp_a, label='Patient A', color='blue', marker='o')
plt.plot(hours, crp_b, label='Patient B', color='red', marker='x')
plt.xlabel('Hours after surgery')
plt.ylabel('CRP (mg/L)')
plt.legend()
plt.show()

### Predict: the x-axis of `df_hr['hr'].plot()`
**The row index, 0 to 23.** `df_hr['hr']` is a Series: it has values and an index, and the index is what pandas plots along x. The times live in a different column the Series knows nothing about. `df_hr.plot('obs_time', 'hr')` names both.

### Predict: plots are opinionated
From the **pie** nobody can say which time of day is highest: the four slices are within a few percent of each other, and a pie of means is a false claim anyway, since the means are not parts of a whole. From the **bar** with its axis starting at 70, afternoon looks double night; the real gap is 4 bpm on a mean of about 75. The **box plots** put the 4 bpm against a spread of about 9 bpm within each group: the differences are real but small, and the groups overlap. Same data, three stories; the plot type and the axis range decided which one you believed.

## Capstone: does recovery depend on who the patient is?
**Approach:** merge on `id` and check the shape; `hist` for the overall distribution; one `groupby` for the means and `.plot(kind='bar')` straight on its result; `boxplot(column, by)` for the distributions; a string slice and a cast for birth year; a scatter plot for the relationship. For the stretch, a function with an `if` / `elif` chain using `code in 'ABC'`, applied to the column with `.apply()`; the groups are the NHS Data Dictionary's own, which is the kind of mapping a real pipeline takes from a code list rather than invents.

**What the plots show:** recovery days are roughly symmetric around 10, and 40 bins show sampling noise rather than extra shape. Mean recovery differs by gender here (Female ≈ 12.7 days, Male ≈ 8.0, Not known ≈ 9.7); the bar chart shows the means, and the box plots show what it hides, that the distributions overlap substantially. Birth year shows no visible relationship with recovery. Grouped ethnicity shows no large differences. None of this is a statistical test: that comes later in the course.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
DATA = "https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/data/"

df_demographics = pd.read_csv(DATA + "demographics.csv")
df_recovery = pd.read_csv(DATA + "simple_recovery.csv")

# 1. Merge and check
df = pd.merge(df_demographics, df_recovery, on='id')
print(df.shape)                                        # (1000, 7)

# 2. Histogram of recovery days, two bin counts
df.hist('recovery_days', bins=10)
plt.show()
df.hist('recovery_days', bins=40)
plt.show()

# 3. Mean recovery by gender, as numbers and as a bar chart
mean_by_gender = df.groupby('gender')['recovery_days'].mean()
print(mean_by_gender)
mean_by_gender.plot(kind='bar')
plt.ylabel('Mean recovery (days)')
plt.show()

# 4. Box plot by gender
df.boxplot(column='recovery_days', by='gender')
plt.suptitle('')
plt.show()

# 5. Birth year and a scatter plot
df['birth_year'] = df['date_of_birth'].str[:4].astype(int)
df.plot('birth_year', 'recovery_days', kind='scatter')
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
DATA = "https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/data/"

df_demographics = pd.read_csv(DATA + "demographics.csv")
df_recovery = pd.read_csv(DATA + "simple_recovery.csv")
df = pd.merge(df_demographics, df_recovery, on='id')

# 6. Stretch: ethnicity codes to groups
def ethnicity_group(code):
    if code in 'ABC':
        return 'White'
    elif code in 'DEFG':
        return 'Mixed'
    elif code in 'HJKL':
        return 'Asian or Asian British'
    elif code in 'MNP':
        return 'Black or Black British'
    elif code in 'RS':
        return 'Other Ethnic Groups'
    else:
        return 'Not stated'

df['ethnicity_group'] = df['ethnicity_code'].apply(ethnicity_group)
print(df['ethnicity_group'].value_counts())

df.boxplot(column='recovery_days', by='ethnicity_group', rot=30)
plt.suptitle('')
plt.show()

## Additional work 1 — Lab tests and recovery
**Approach:** the lab table's `id` is the test id, so the merge key is `patient_id` against recovery's `id`, giving 3000 rows (three tests per patient). Both tables have a column called `id`, so pandas keeps both as `id_x` and `id_y`. Pivoting on `patient_id` and `recovery_days` together keeps recovery as a column. Each test then needs its own cleaning: a cast for WBC; a split, a cast and a unit conversion for CRP; lower-casing for albumin.

**What the plots show:** recovery rises with WBC z-score and with CRP. Binning CRP and stratifying by gender shows the CRP association is largely explained by gender (little variation within each gender), while the WBC association holds within each gender. Albumin labels, once lower-cased, give five groups with modest differences.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
DATA = "https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/data/"

df_lab = pd.read_csv(DATA + "laboratory_tests.csv")
df_recovery = pd.read_csv(DATA + "simple_recovery.csv")
df_demographics = pd.read_csv(DATA + "demographics.csv")

# 1. Merge on differently named keys
df_long = pd.merge(df_lab, df_recovery, left_on='patient_id', right_on='id')
print(df_long.shape)                                   # (3000, 6): both id columns kept, as id_x and id_y

# 2. Pivot to one row per patient
df_wide = df_long.pivot(index=['patient_id', 'recovery_days'], columns='test', values='value').reset_index()
print(df_wide.head())

# 3. WBC z-score: cast and scatter
df_wide['wbc_z_score'] = df_wide['wbc_z_score'].astype(float)
df_wide.plot('wbc_z_score', 'recovery_days', kind='scatter')
plt.show()

# 4. CRP: split value and unit, convert g/L to mg/L
crp = df_wide['crp'].str.split(' ', expand=True).rename(columns={0: 'value', 1: 'unit'})
crp['value'] = crp['value'].astype(float)
is_gl = crp['unit'] == 'g/L'
crp.loc[is_gl, 'value'] = crp.loc[is_gl, 'value'] * 1000
df_wide['crp_mg_l'] = crp['value']
df_wide.plot('crp_mg_l', 'recovery_days', kind='scatter')
plt.show()

# 5. Albumin: lower-case the labels, then box plot
print(df_wide['albumin_level'].unique())
df_wide['albumin_level'] = df_wide['albumin_level'].str.lower()
df_wide.boxplot(column='recovery_days', by='albumin_level')
plt.suptitle('')
plt.show()

# 6. Stretch: bin CRP and stratify by gender
df_wide['crp_bins'] = pd.cut(df_wide['crp_mg_l'], [0, 0.5, 1, 1.5, 2, 2.5])
df_all = pd.merge(df_wide, df_demographics[['id', 'gender']], left_on='patient_id', right_on='id')
df_all.boxplot(column='recovery_days', by=['gender', 'crp_bins'], rot=90)
plt.suptitle('')
plt.show()

## Additional work 2 — Merge types
### Predict: a left merge
**`(1000, 7)` and `990`.** `how='left'` keeps every row of the left table (demographics), so all thousand patients are back; the 990 without a match in the ten-row sample get `NaN` in `recovery_days`. `isna()` gives `True` for those and the sum counts them. This is the "if nothing found" column of a feature card, made concrete: a left join and a `NaN`.

### Your Turn: right, outer, and the full table
`right` keeps every row of the sample: `(10, 7)`. `outer` keeps every key from either side: `(1000, 7)`, the same as `left` here because every sample key is also in demographics. With the whole recovery table all four types give `(1000, 7)`, because every patient appears in both tables exactly once, so there is nothing for the type to decide. The type only matters when keys are missing from one side or repeated.

In [ ]:
import pandas as pd
DATA = "https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/data/"

df_demographics = pd.read_csv(DATA + "demographics.csv")
df_recovery = pd.read_csv(DATA + "simple_recovery.csv")
recovery_sample = df_recovery.head(10)

for how in ['inner', 'left', 'right', 'outer']:
    print(how, pd.merge(df_demographics, recovery_sample, on='id', how=how).shape,
          pd.merge(df_demographics, df_recovery, on='id', how=how).shape)

## Additional work 3 — Subplots and saving
**Approach:** two panels with `subplot(1, 2, 1)` and `subplot(1, 2, 2)`; `savefig` before `show`, because `show` clears the figure.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
DATA = "https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/data/"

df_hr = pd.read_csv(DATA + 'hr_timeseries.csv')

plt.subplot(1, 2, 1)
plt.plot(df_hr['hr'])
plt.title('Heart rate')

plt.subplot(1, 2, 2)
plt.hist(df_hr['hr'], bins=8)
plt.title('Distribution')

plt.tight_layout()
plt.savefig('hr.png')
plt.show()

## Additional work 4 — Reshaping

### Predict: `pivot` with a repeated observation
**It refuses**, with `ValueError: Index contains duplicate entries, cannot reshape`. A pivot puts one value in each cell and cannot choose between two. `pivot_table` with `aggfunc` is the version that decides for you.

### Quick check
1. `pd.concat`: stacking tables with the same columns.
2. One row per **subject**, one column per observation.
3. `pivot` (or `pivot_table` if observations repeat).
4. `pd.merge`, on the shared key.

## Additional work 5 — Regular expressions

### Predict: `str.contains('+')`
**An error.** In a regex `+` means "one or more of the preceding thing", and there is no preceding thing, so the pattern is invalid (the message is `nothing to repeat` or `no argument for repetition operator`, depending on the pandas version). Escape it, `r'\+'`, or pass `regex=False`.